# 🤖 AI Data Analyst Agent
An autonomous data analysis agent powered by Google Gemini.

**What this agent does:**
- Accepts any CSV dataset
- Inspects the data and identifies the problem type
- Selects the appropriate ML model (Logistic Regression, Random Forest, Linear Regression)
- Reasons about hyperparameters
- Trains, evaluates, and visualizes results
- Produces a plain-English summary of findings

**Stack:** Python · Google Gemini 2.5 Flash · scikit-learn · pandas · matplotlib · seaborn

## 1. Setup & Imports

In [2]:
# Install required packages
!pip install google-genai pandas matplotlib seaborn scikit-learn -q

# Core imports
from google import genai
from google.genai import types
from google.colab import files
from IPython.display import display, Image
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import io
import sys
import os

# ML imports
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    mean_squared_error, mean_absolute_error, r2_score
)
from sklearn.preprocessing import LabelEncoder, StandardScaler

print('✅ All imports successful!')

✅ All imports successful!


## 2. API Key Configuration

> ⚠️ **Never share your API key publicly.**
> Store it in Colab Secrets (🔑 icon in left sidebar) with the name `GEMINI_API_KEY`,
> or paste it directly below for local testing only.

In [3]:
# Option A: Load from Colab Secrets (recommended)
# from google.colab import userdata
# GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

# Option B: Paste key directly (for local testing only — never commit to GitHub)
GEMINI_API_KEY = "Ayour-api-key-here"

client = genai.Client(api_key=GEMINI_API_KEY)
print('✅ Gemini client ready!')

✅ Gemini client ready!


## 3. Agent System Prompt
Defines the agent's persona, workflow, and rules.

In [4]:
SYSTEM_INSTRUCTION = """
You are an expert data scientist agent.
You are given a dataset and a goal. You reason step by step like a senior data scientist would.

## Your workflow:
1. INSPECT the data — column types, nulls, distributions, target variable
2. IDENTIFY the problem type based on the target variable:
   - Binary column (0/1, yes/no) → Logistic Regression + Random Forest (classification)
   - Continuous numeric column → Linear Regression (regression)
   - Categorical column with 3+ classes → Random Forest (multiclass classification)
   - No clear target → K-Means Clustering (unsupervised)
3. SELECT the best model and explain why
4. CONSIDER hyperparameters — don't just use defaults, reason about:
   - For Logistic Regression: C (regularization strength), max_iter, solver
   - For Random Forest: n_estimators, max_depth, min_samples_split
   - For Linear Regression: whether to normalize, handle outliers
   - For K-Means: n_clusters (use elbow method)
5. TRAIN the model using an 80/20 train/test split
6. EVALUATE using the right metrics:
   - Classification → accuracy, precision, recall, F1, confusion matrix
   - Regression → RMSE, MAE, R² score
7. VISUALIZE results — feature importance, confusion matrix, residual plots
8. SUMMARIZE findings in plain English

## Rules:
- Always wrap code in ```python ... ``` blocks
- Always save charts with plt.savefig('chart.png', bbox_inches='tight'); plt.close()
- Never use plt.show()
- Encode categorical variables before modeling (use pd.get_dummies or LabelEncoder)
- Always print evaluation metrics clearly
- When done, start your final message with: FINAL ANSWER:
"""

print('✅ System prompt ready!')

✅ System prompt ready!


## 4. Core Agent Functions

In [5]:
# --- ML tools available to the agent ---
def get_ml_tools(df):
    """Returns a dict of all tools the agent can use when executing code."""
    return {
        "pd": pd, "plt": plt, "sns": sns, "os": os, "np": np,
        "df": df,
        "LogisticRegression": LogisticRegression,
        "LinearRegression": LinearRegression,
        "RandomForestClassifier": RandomForestClassifier,
        "RandomForestRegressor": RandomForestRegressor,
        "KMeans": KMeans,
        "train_test_split": train_test_split,
        "GridSearchCV": GridSearchCV,
        "accuracy_score": accuracy_score,
        "precision_score": precision_score,
        "recall_score": recall_score,
        "f1_score": f1_score,
        "confusion_matrix": confusion_matrix,
        "classification_report": classification_report,
        "mean_squared_error": mean_squared_error,
        "mean_absolute_error": mean_absolute_error,
        "r2_score": r2_score,
        "LabelEncoder": LabelEncoder,
        "StandardScaler": StandardScaler,
    }


# --- Code execution tool ---
def run_python(code: str, df) -> str:
    """Executes Python code written by the agent and returns printed output."""
    old_stdout = sys.stdout
    sys.stdout = buffer = io.StringIO()
    try:
        exec(code, get_ml_tools(df))

        # Force-save any still-open matplotlib figures
        for i, fig in enumerate(map(plt.figure, plt.get_fignums())):
            fig.savefig(f'chart_{i}.png', bbox_inches='tight')
            plt.close(fig)

        output = buffer.getvalue()

    except Exception as e:
        output = f"ERROR: {str(e)}"

    finally:
        # Restore stdout BEFORE displaying images
        sys.stdout = old_stdout

    # Display any saved charts
    saved_charts = [f for f in os.listdir('.') if f.endswith('.png')]
    for fname in saved_charts:
        display(Image(filename=fname))
        output += f"\n[Chart '{fname}' displayed]"
    for fname in saved_charts:
        os.remove(fname)

    return output if output else "Code ran successfully (no printed output)."


# --- Message helpers ---
def user_msg(text):
    return types.Content(role="user", parts=[types.Part(text=text)])

def model_msg(text):
    return types.Content(role="model", parts=[types.Part(text=text)])


# --- Dataset summary (avoids sending full data to model) ---
def get_dataset_summary(df):
    """Sends only a sample + stats to the model instead of the full dataset."""
    return f"""
Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns

Column names and types:
{df.dtypes.to_string()}

First 5 rows:
{df.head().to_string()}

Descriptive statistics:
{df.describe().to_string()}
"""


# --- Main agent loop ---
def run_agent(goal: str, df, max_turns: int = 8):
    """Runs the AI agent on a given dataframe and goal."""
    print(f"🎯 Goal: {goal}\n")
    print("=" * 50)

    dataset_context = f"""
Here is the dataset you will be analyzing (as a pandas DataFrame called 'df'):

{get_dataset_summary(df)}

Your goal: {goal}

Start by reasoning about what steps to take, then write Python code to begin.
"""

    messages = [user_msg(dataset_context)]

    for turn in range(max_turns):
        print(f"\n🤖 Agent turn {turn + 1}...")

        response = client.models.generate_content(
            model="gemini-2.5-flash",
            config=types.GenerateContentConfig(system_instruction=SYSTEM_INSTRUCTION),
            contents=messages
        )

        reply = response.text
        print(reply)
        messages.append(model_msg(reply))

        # Run any code in the final answer before stopping
        if "FINAL ANSWER:" in reply:
            if "```python" in reply:
                code_block = reply.split("```python")[1].split("```")[0].strip()
                run_python(code_block, df)
            print("\n✅ Agent finished!")
            break

        if "```python" in reply:
            code_block = reply.split("```python")[1].split("```")[0].strip()
            print("\n⚙️ Running code...")
            result = run_python(code_block, df)
            print(f"📊 Result:\n{result}")
            messages.append(user_msg(f"Code output:\n{result}\n\nContinue your analysis."))
        else:
            messages.append(user_msg("Continue your analysis."))

    return messages


print('✅ Agent functions ready!')

✅ Agent functions ready!


## 5. Upload & Load Dataset
Upload any CSV file. The agent will inspect it and determine the best analysis approach.

In [6]:
def load_dataset():
    """Prompts user to upload a CSV and loads it as a DataFrame."""
    print("📂 Upload a CSV file...")
    uploaded = files.upload()

    if not uploaded:
        print("No file uploaded.")
        return None

    fname = list(uploaded.keys())[0]
    df = pd.read_csv(io.BytesIO(uploaded[fname]))

    print(f"\n✅ Loaded '{fname}' — {df.shape[0]} rows x {df.shape[1]} columns")
    print("\n--- Preview ---")
    print(df.head())
    print("\n--- Column Types ---")
    print(df.dtypes)

    return df

df = load_dataset()

📂 Upload a CSV file...


Saving customer_data.csv to customer_data.csv

✅ Loaded 'customer_data.csv' — 100 rows x 14 columns

--- Preview ---
  customer_id  age gender region  tenure_months  num_purchases  \
0        C001   34      M  North             24             12   
1        C002   52      F  South              6              3   
2        C003   28      F   East             36             20   
3        C004   45      M   West             12              7   
4        C005   61      F  North              3              2   

   avg_order_value  last_purchase_days_ago  support_tickets product_category  \
0             85.5                      15                1      Electronics   
1            120.0                      45                3         Clothing   
2             65.0                       7                0             Food   
3            200.0                      30                2      Electronics   
4             45.0                      60                5         Clothing   

   to

## 6. Run the Agent
The agent will inspect the data, select appropriate ML models, train and evaluate them, generate charts, and summarize findings.

In [7]:
run_agent(
    goal="""Analyze this customer dataset.
    Identify the best ML models to run based on the data,
    select appropriate hyperparameters, train and evaluate the models,
    generate visualizations, and summarize your findings.""",
    df=df
);

🎯 Goal: Analyze this customer dataset.
    Identify the best ML models to run based on the data,
    select appropriate hyperparameters, train and evaluate the models,
    generate visualizations, and summarize your findings.


🤖 Agent turn 1...
The goal is to analyze the provided customer dataset to predict customer churn, which is a binary classification problem.

Here's the plan:

1.  **Inspect Data**: Check for missing values, data types, and unique values in categorical columns. The 'churned' column is our target variable.
2.  **Identify Problem Type**: Since 'churned' is a binary column (0 or 1), this is a **binary classification** problem.
3.  **Select Models**:
    *   **Logistic Regression**: A good baseline for classification, interpretable.
    *   **Random Forest Classifier**: Robust and often high-performing, can capture non-linear relationships and interactions.
4.  **Data Preprocessing**:
    *   Drop `customer_id` as it's an identifier.
    *   Convert categorical featu